# Demo: Supervised learning

## Instituto Superior Técnico, University of Lisbon

**Problem statement**: Classify the MNIST hand-written digits dataset.

Author: Rodrigo Ventura (<rodrigo.ventura@tecnico.ulisboa.pt>)

In [ ]:
import time
import numpy as np
import sklearn

%matplotlib widget
import matplotlib.pyplot as plt

## Dataset loading ##

Origin of the data is UCI ML hand-written digits dataset. The dataset is built-in with [scikit-learn](https://scikit-learn.org/).

In [ ]:
dataset = sklearn.datasets.fetch_openml("mnist_784", as_frame=False)

Xtrain, Xtest, ytrain, ytest = sklearn.model_selection.train_test_split(
    dataset.data, dataset.target,
    test_size=0.2)

print("Dimension of train set:", Xtrain.shape)
print("Dimension of test set:", Xtest.shape)

img_shape = (28, 28)  # hence 784 features (i.e., pixels)

Shows some of the dataset instances in a grid.

In [ ]:
H, W = 5, 5
fig,axs = plt.subplots(H, W)
k = 0
for i in range(H):
    for j in range(W):
        axs[i,j].imshow(dataset['data'][k].reshape((28,28)), cmap='gray')
        axs[i,j].axis('off')
        k += 1

## Supervised learning ##

In [ ]:
# This function computes the accuracy of a model on a given dataset

def show_evaluation(model, X, y):
    # Testing is done here:
    out = model.predict(X)
    confusion = sklearn.metrics.confusion_matrix(y, out)
    print("Confusion matrix:")
    print(confusion)
    acc = 100 * np.sum(y==out) / len(y)
    print(f"Accuracy: {acc:.2f}%")

### k Nearest Neighbors (k-NN) classifier ###

In [ ]:
# Training is done here:
model = sklearn.neighbors.KNeighborsClassifier(n_neighbors=1)
model.fit(Xtrain, ytrain)

# Limit the evaluation belows to N instances (otherwise it'd be way too slow)
N = 1000

print(f"----- PERFORMANCE ON {N} INSTANCES OF THE TRAIN SET -----")
show_evaluation(model, Xtrain[:N], ytrain[:N])

print("\n----- PERFORMANCE ON {N} INSTANCES OF THE TEST SET -----")
show_evaluation(model, Xtest[:N], ytest[:N])

### Support Vector Machine (SVM) classifier ###

First, let's normalize train set to zero mean and unit variance.

In [ ]:
scaler = sklearn.preprocessing.StandardScaler()
scaler.fit(Xtrain)  # Compute normalization from train set only, to prevent data leakage

Xtrain = scaler.transform(Xtrain)
Xtest = scaler.transform(Xtest)

Next cell performs training and test of a SVM classifier.

In [ ]:
# Since SVM training is very slow with the raw images as input,
# limit train instances,
Ntrain = 10000
# while in testing, the full dataset can be used

# Setup the classifier -- choose one by commenting all of the other options
model = sklearn.svm.SVC(kernel='linear')
#model = sklearn.svm.SVC(kernel='rbf')
#model = sklearn.svm.SVC(kernel='rbf', C=5, gamma=0.05)  # poor performance -- WiP


# Training is done here:
t0 = time.perf_counter()
model.fit(Xtrain[:Ntrain], ytrain[:Ntrain])
t1 = time.perf_counter()
print(f"Training took {t1-t0:.3f} seconds.\n")

print(f"----- PERFORMANCE ON THE TRAIN SUBSET USED FOR TRAINING: {Ntrain} instances -----")
show_evaluation(model, Xtrain[:Ntrain], ytrain[:Ntrain])

#print(f"\n----- PERFORMANCE ON THE ORIGINAL TRAIN SET: {len(Xtrain)} instances -----")
#show_evaluation(model, Xtrain, ytrain)

print(f"\n----- PERFORMANCE ON THE TEST SET: {len(Xtest)} instances -----")
show_evaluation(model, Xtest, ytest)